In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
import optuna
import os

# データディレクトリの設定
data_dir = r"c:\Users\mikku\OneDrive\デスクトップ\機械学習のお勉強\003Kaggle\Diabates-Prediction\data"

In [3]:
# データの読み込み
train = pd.read_csv(os.path.join(data_dir, "train.csv"))
test = pd.read_csv(os.path.join(data_dir, "test.csv"))

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("\nTrain columns:", train.columns.tolist())
train.head(2)

Train shape: (700000, 26)
Test shape: (300000, 25)

Train columns: ['id', 'age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides', 'gender', 'ethnicity', 'education_level', 'income_level', 'smoking_status', 'employment_status', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history', 'diagnosed_diabetes']


,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
0,0,31,1,45,7.7,6.8,6.1,33.4,0.93,112,...,Female,Hispanic,Highschool,Lower-Middle,Current,Employed,0,0,0,1.0
1,1,50,2,73,5.7,6.5,5.8,23.8,0.83,120,...,Female,White,Highschool,Upper-Middle,Never,Employed,0,0,0,1.0


In [4]:
# カテゴリ変数の処理
# カテゴリ変数をLabel Encodingで数値化
categorical_cols = ['gender', 'ethnicity', 'education_level', 'income_level', 'smoking_status', 'employment_status']

le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])
    le_dict[col] = le

print("カテゴリ変数を数値化しました")
train.head(2)

カテゴリ変数を数値化しました


,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
0,0,31,1,45,7.7,6.8,6.1,33.4,0.93,112,...,0,2,1,2,0,0,0,0,0,1.0
1,1,50,2,73,5.7,6.5,5.8,23.8,0.83,120,...,0,4,1,4,2,0,0,0,0,1.0


In [5]:
# 特徴量とターゲットの分離
X = train.drop(['id', 'diagnosed_diabetes'], axis=1)
y = train['diagnosed_diabetes']

# 訓練データとバリデーションデータに分割
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_val shape: {y_val.shape}")

X_train shape: (560000, 24)
X_val shape: (140000, 24)
y_train shape: (560000,)
y_val shape: (140000,)


In [6]:
# モデルのトレーニング（ロジスティック回帰）
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

print("モデルのトレーニングが完了しました")

モデルのトレーニングが完了しました


c:\Users\mikku\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [7]:
# モデルのトレーニング（XGBoost）
xgb_model = XGBClassifier(random_state=42, n_estimators=100, max_depth=6, learning_rate=0.1, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

print("XGBoostのトレーニングが完了しました")

XGBoostのトレーニングが完了しました


In [8]:
# XGBoostモデルでの予測と評価
y_val_pred_xgb = xgb_model.predict(X_val)
y_val_pred_proba_xgb = xgb_model.predict_proba(X_val)[:, 1]

# 評価指標の計算
accuracy_xgb = accuracy_score(y_val, y_val_pred_xgb)
precision_xgb = precision_score(y_val, y_val_pred_xgb)
recall_xgb = recall_score(y_val, y_val_pred_xgb)
f1_xgb = f1_score(y_val, y_val_pred_xgb)
auc_xgb = roc_auc_score(y_val, y_val_pred_proba_xgb)

print("=" * 50)
print("XGBoost バリデーション結果")
print("=" * 50)
print(f"Accuracy:  {accuracy_xgb:.4f}")
print(f"Precision: {precision_xgb:.4f}")
print(f"Recall:    {recall_xgb:.4f}")
print(f"F1-Score:  {f1_xgb:.4f}")
print(f"AUC-ROC:   {auc_xgb:.4f}")
print("=" * 50)

XGBoost バリデーション結果
Accuracy:  0.6775
Precision: 0.6975
Recall:    0.8533
F1-Score:  0.7676
AUC-ROC:   0.7187


In [ ]:
# バリデーションデータでの予測と評価
y_val_pred = model.predict(X_val)
y_val_pred_proba = model.predict_proba(X_val)[:, 1]

# 評価指標の計算
accuracy = accuracy_score(y_val, y_val_pred)
precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)
auc = roc_auc_score(y_val, y_val_pred_proba)

print("=" * 50)
print("バリデーション結果（テストデータ）")
print("=" * 50)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"AUC-ROC:   {auc:.4f}")
print("=" * 50)

バリデーション結果
Accuracy:  0.6612
Precision: 0.6792
Recall:    0.8661
F1-Score:  0.7614
AUC-ROC:   0.6940


In [10]:
# Optunaでハイパーパラメータ最適化（クロスバリデーション付き）
def objective(trial):
    # ハイパーパラメータの探索範囲を定義
    # .trialはOptunaのTrialオブジェクトで、ハイパーパラメータの提案を行うために使用されます。
    # 各パラメータは
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    
    # XGBoostモデルの作成
    model = XGBClassifier(**params)
    
    # 5分割クロスバリデーションでAUC-ROCを評価
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
    
    return scores.mean()

# Optunaで最適化実行
print("Optunaでハイパーパラメータ最適化を開始します...")
print("これには数分かかる場合があります。")

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("\n最適化完了！")
print(f"ベストスコア (AUC-ROC): {study.best_value:.4f}")
print(f"ベストパラメータ: {study.best_params}")

[I 2025-12-24 19:06:51,862] A new study created in memory with name: no-name-14c80a11-37d6-4d49-8930-8d3ca8f8f626


Optunaでハイパーパラメータ最適化を開始します...
これには数分かかる場合があります。


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-12-24 19:07:04,479] Trial 0 finished with value: 0.7108977821444473 and parameters: {'n_estimators': 144, 'max_depth': 10, 'learning_rate': 0.22227824312530747, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'gamma': 0.02904180608409973}. Best is trial 0 with value: 0.7108977821444473.
[I 2025-12-24 19:07:20,955] Trial 1 finished with value: 0.7162970486513036 and parameters: {'n_estimators': 267, 'max_depth': 7, 'learning_rate': 0.21534104756085318, 'subsample': 0.608233797718321, 'colsample_bytree': 0.9879639408647978, 'min_child_weight': 9, 'gamma': 0.10616955533913808}. Best is trial 1 with value: 0.7162970486513036.
[I 2025-12-24 19:07:27,054] Trial 2 finished with value: 0.7115085561931913 and parameters: {'n_estimators': 95, 'max_depth': 4, 'learning_rate': 0.09823025045826593, 'subsample': 0.8099025726528951, 'colsample_bytree': 0.7727780074568463, 'min_child_weight': 3, 'gamma': 0.30592644736118974}. Best is trial 1 with

In [11]:
# 最適化されたパラメータでモデルを再トレーニング
best_params = study.best_params
best_params['random_state'] = 42
best_params['eval_metric'] = 'logloss'

xgb_optimized = XGBClassifier(**best_params)
xgb_optimized.fit(X_train, y_train)

print("最適化されたXGBoostモデルのトレーニングが完了しました")

最適化されたXGBoostモデルのトレーニングが完了しました


In [12]:
# 最適化モデルでの予測と評価
y_val_pred_opt = xgb_optimized.predict(X_val)
y_val_pred_proba_opt = xgb_optimized.predict_proba(X_val)[:, 1]

# 評価指標の計算
accuracy_opt = accuracy_score(y_val, y_val_pred_opt)
precision_opt = precision_score(y_val, y_val_pred_opt)
recall_opt = recall_score(y_val, y_val_pred_opt)
f1_opt = f1_score(y_val, y_val_pred_opt)
auc_opt = roc_auc_score(y_val, y_val_pred_proba_opt)

print("=" * 50)
print("最適化XGBoost バリデーション結果")
print("=" * 50)
print(f"Accuracy:  {accuracy_opt:.4f}")
print(f"Precision: {precision_opt:.4f}")
print(f"Recall:    {recall_opt:.4f}")
print(f"F1-Score:  {f1_opt:.4f}")
print(f"AUC-ROC:   {auc_opt:.4f}")
print("=" * 50)

最適化XGBoost バリデーション結果
Accuracy:  0.6831
Precision: 0.7057
Recall:    0.8443
F1-Score:  0.7688
AUC-ROC:   0.7250


In [13]:
# 全モデルの比較
print("\n" + "=" * 70)
print("全モデル比較")
print("=" * 70)
print(f"{'指標':<15} {'ロジスティック回帰':<20} {'XGBoost':<20} {'最適化XGBoost':<20}")
print("-" * 70)
print(f"{'Accuracy':<15} {accuracy:<20.4f} {accuracy_xgb:<20.4f} {accuracy_opt:<20.4f}")
print(f"{'Precision':<15} {precision:<20.4f} {precision_xgb:<20.4f} {precision_opt:<20.4f}")
print(f"{'Recall':<15} {recall:<20.4f} {recall_xgb:<20.4f} {recall_opt:<20.4f}")
print(f"{'F1-Score':<15} {f1:<20.4f} {f1_xgb:<20.4f} {f1_opt:<20.4f}")
print(f"{'AUC-ROC':<15} {auc:<20.4f} {auc_xgb:<20.4f} {auc_opt:<20.4f}")
print("=" * 70)

# 改善率の計算
print("\n改善率（ロジスティック回帰からの改善）:")
print(f"XGBoost:          {(accuracy_xgb - accuracy) / accuracy * 100:+.2f}%")
print(f"最適化XGBoost:    {(accuracy_opt - accuracy) / accuracy * 100:+.2f}%")


全モデル比較
指標              ロジスティック回帰            XGBoost              最適化XGBoost          
----------------------------------------------------------------------
Accuracy        0.6612               0.6775               0.6831              
Precision       0.6792               0.6975               0.7057              
Recall          0.8661               0.8533               0.8443              
F1-Score        0.7614               0.7676               0.7688              
AUC-ROC         0.6940               0.7187               0.7250              

改善率（ロジスティック回帰からの改善）:
XGBoost:          +2.46%
最適化XGBoost:    +3.32%


In [14]:
# テストデータに対する予測（最適化XGBoostを使用）
test_X = test.drop('id', axis=1)
test_pred_optimized = xgb_optimized.predict_proba(test_X)[:, 1]

# 提出ファイルの生成
submission_optimized = pd.DataFrame({
    'id': test['id'],
    'diagnosed_diabetes': test_pred_optimized
})

# CSVファイルとして保存
output_path = 'submission_xgboost_optimized.csv'
submission_optimized.to_csv(output_path, index=False)

print(f"\n提出ファイルを生成しました: {output_path}")
print(submission_optimized.head())
print(f"\n提出ファイルの形状: {submission_optimized.shape}")


提出ファイルを生成しました: submission_xgboost_optimized.csv
       id  diagnosed_diabetes
0  700000            0.531986
1  700001            0.672672
2  700002            0.783511
3  700003            0.401775
4  700004            0.922600

提出ファイルの形状: (300000, 2)
